In [3]:
import pandas as pd
import sqlite3

# Load CSV
df = pd.read_excel("Telco_customer_churn.xlsx")

# Create SQLite database
conn = sqlite3.connect("telecom_churn.db")

# Load data into SQL
df.to_sql("telco_raw", conn, if_exists="replace", index=False)

conn.close()


In [4]:
conn = sqlite3.connect("telecom_churn.db") 
query = "SELECT COUNT(*) FROM telco_raw;"
cursor = conn.cursor()
pd.read_sql(query, conn)


,COUNT(*)
0,7043


In [7]:
cursor.execute(""" CREATE TABLE customers ( 
                customer_id TEXT PRIMARY KEY,
                gender TEXT,
                senior_citizen INTEGER,
                partner TEXT,
                dependents TEXT ); """)

OperationalError: table customers already exists

In [ ]:
cursor.execute(""" CREATE TABLE subscriptions (
               customer_id TEXT,
               tenure INTEGER,
               contract_type TEXT,
               payment_method TEXT,
               paperless_billing TEXT,
               FOREIGN KEY (customer_id) REFERENCES customers (customer_id)); """)

In [ ]:
cursor.execute("""CREATE TABLE services (
    customer_id TEXT,
    internet_service TEXT,
    online_security TEXT,
    tech_support TEXT,
    streaming_tv TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id));""")

In [12]:
cursor.execute("""CREATE TABLE IF NOT EXISTS charges (
    customer_id TEXT,
    monthly_charges REAL,
    total_charges REAL,
    churn INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id));""")

In [ ]:
cursor.execute(""" INSERT INTO customers
SELECT
    customerID,
    gender,
    [Senior Citizen],
    Partner,
    Dependents
FROM telco_raw;""")
conn.commit()

In [ ]:
import sqlite3

conn = sqlite3.connect("telecom_churn.db")
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(telco_raw);")
columns = cursor.fetchall()

for col in columns:
    print(col)




(0, 'CustomerID', 'TEXT', 0, None, 0)
(1, 'Count', 'INTEGER', 0, None, 0)
(2, 'Country', 'TEXT', 0, None, 0)
(3, 'State', 'TEXT', 0, None, 0)
(4, 'City', 'TEXT', 0, None, 0)
(5, 'Zip Code', 'INTEGER', 0, None, 0)
(6, 'Lat Long', 'TEXT', 0, None, 0)
(7, 'Latitude', 'REAL', 0, None, 0)
(8, 'Longitude', 'REAL', 0, None, 0)
(9, 'Gender', 'TEXT', 0, None, 0)
(10, 'Senior Citizen', 'TEXT', 0, None, 0)
(11, 'Partner', 'TEXT', 0, None, 0)
(12, 'Dependents', 'TEXT', 0, None, 0)
(13, 'Tenure Months', 'INTEGER', 0, None, 0)
(14, 'Phone Service', 'TEXT', 0, None, 0)
(15, 'Multiple Lines', 'TEXT', 0, None, 0)
(16, 'Internet Service', 'TEXT', 0, None, 0)
(17, 'Online Security', 'TEXT', 0, None, 0)
(18, 'Online Backup', 'TEXT', 0, None, 0)
(19, 'Device Protection', 'TEXT', 0, None, 0)
(20, 'Tech Support', 'TEXT', 0, None, 0)
(21, 'Streaming TV', 'TEXT', 0, None, 0)
(22, 'Streaming Movies', 'TEXT', 0, None, 0)
(23, 'Contract', 'TEXT', 0, None, 0)
(24, 'Paperless Billing', 'TEXT', 0, None, 0)
(25, 'Pay

In [ ]:
cursor.execute(""" INSERT INTO subscriptions 
               SELECT 
               customerID,
               [Tenure Months],
               contract,
               [Payment Method],
               [Paperless Billing]
               FROM telco_raw;""")
conn.commit()

In [ ]:
cursor.execute(""" INSERT INTO services
SELECT
    customerID,
    [Internet Service],
    [Online Security],
    [Tech Support],
    [Streaming TV]
FROM telco_raw;""")
conn.commit()

In [10]:
cursor.execute(""" INSERT INTO charges
SELECT
    customerID,
    [Monthly Charges],
    [Total Charges],
    [Churn Value]
FROM telco_raw;
""")
conn.commit()

In [ ]:
cursor.execute("""SELECT COUNT(*) FROM customers;""")
result=cursor.fetchone()
print (result[0])

7043


In [ ]:
cursor.execute("""SELECT COUNT(*) FROM subscriptions;""")
result=cursor.fetchone()
print (result[0])

7043


In [ ]:
cursor.execute("""SELECT COUNT(*) FROM services;""")
result=cursor.fetchone()
print (result[0])

7043


In [ ]:
cursor.execute("""SELECT COUNT(*) FROM charges;""")
result=cursor.fetchone()
print (result[0])

7043


In [3]:
cursor.execute("""
SELECT
    s.contract_type,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN c.churn = 1 THEN 1 ELSE 0 END
) AS churned_customers,
    ROUND(
        1.0 * SUM(CASE WHEN c.churn = 1 THEN 1 ELSE 0 END) / COUNT(*),
        3
    ) AS churn_rate
FROM subscriptions s
JOIN charges c
    ON s.customer_id = c.customer_id
GROUP BY s.contract_type
ORDER BY churn_rate DESC;
""")

rows = cursor.fetchall()
for r in rows:
    print(r)


('Month-to-month', 7750, 1655, 0.214)
('One year', 2946, 166, 0.056)
('Two year', 3390, 48, 0.014)


In [11]:
cursor.execute(""" SELECT sv.tech_support,
               COUNT(*) AS total_customer,
               SUM(CASE WHEN c.churn=1 THEN 1 ELSE 0 END  )AS churned_customers,
               ROUND (
               1.0 * SUM(CASE WHEN c.churn=1 THEN 1 ELSE 0 END ) / COUNT (*) ,3) AS churn_rate
               FROM services sv
               JOIN charges c
               ON sv.customer_id=c.customer_id 
               GROUP BY sv.tech_support
               ORDER BY churn_rate DESC;""")
rows = cursor.fetchall()
for r in rows:
    print(r)

('No', 6946, 1446, 0.208)
('Yes', 4088, 310, 0.076)
('No internet service', 3052, 113, 0.037)


In [12]:
cursor.execute(""" SELECT sv.online_security,
               COUNT(*) AS total_customer,
               SUM(CASE WHEN c.churn=1 THEN 1 ELSE 0 END) AS churned_customers, 
               ROUND (
               1.0 * SUM (CASE WHEN c.churn=1 THEN 1 ELSE 0 END) / COUNT (*),3) AS churn_rate
               FROM services sv
               JOIN charges c 
               ON sv.customer_id=c.customer_id
               GROUP BY sv.online_security
               ORDER BY churn_rate DESC;""")
rows = cursor.fetchall()
for r in rows:
    print(r)
    

('No', 6996, 1461, 0.209)
('Yes', 4038, 295, 0.073)
('No internet service', 3052, 113, 0.037)


In [5]:
cursor.execute("""SELECT 
               CASE
               WHEN s.tenure <=6 THEN '0-6 Months'
               WHEN s.tenure <=12 THEN '6-12 Months'
               WHEN s.tenure <=24 THEN '1-2 Years'
               ELSE '2+ Years'
               END AS tenure_group,
               COUNT (*) AS total_customer ,
               SUM(CASE WHEN c.churn = 1 THEN 1 ELSE 0 END ) AS churned_customer ,
               ROUND (
               1.0 * SUM(CASE WHEN c.churn = 1 THEN 1 ELSE 0 END )/ COUNT(*),3)AS churn_rate
               FROM subscriptions s
               JOIN charges c 
               ON s.customer_id=c.customer_id
               GROUP BY tenure_group
               ORDER BY churn_rate DESC;
               """)
rows =cursor.fetchall()
for r in rows:
    print(r)

('0-6 Months', 2962, 784, 0.265)
('6-12 Months', 1410, 253, 0.179)
('1-2 Years', 2048, 294, 0.144)
('2+ Years', 7666, 538, 0.07)
